# Homework 6: Decision Trees and Ensemble Learning

Questions: [ML Zoomcamp 2026 – Homework 6](https://github.com/DataTalksClub/machine-learning-zoomcamp/blob/main/cohorts/2026/homework/06-trees/homework.md)

Dataset: [2026 Car Fuel Efficiency](https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv)

In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction import DictVectorizer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, export_text

## Data preparation

Fill missing values with zeros.

In [2]:
target = "fuel_efficiency_mpg"

df = pd.read_csv("data/car_fuel_efficiency_2026.csv")
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [3]:
df.isnull().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

In [4]:
df = df.fillna(0)
df.isnull().sum().sum()

np.int64(0)

## Split the data

Use the exact `train_test_split` calls from the homework (60/20/20, `random_state=1`), then vectorize with `DictVectorizer(sparse=True)`.

In [5]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

y_train = df_train[target].values
y_val = df_val[target].values

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(df_train.drop(columns=target).to_dict(orient="records"))
X_val = dv.transform(df_val.drop(columns=target).to_dict(orient="records"))
features = list(dv.get_feature_names_out())

len(df_train), len(df_val), len(df_test)

(6000, 2000, 2000)

In [6]:
def rmse(y, y_pred):
    return np.sqrt(np.mean((y - y_pred) ** 2))

## Question 1

Train a decision tree regressor with `max_depth=1`. Which original feature is used for splitting?

In [7]:
dt = DecisionTreeRegressor(max_depth=1).fit(X_train, y_train)
print(export_text(dt, feature_names=features))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



In [8]:
features[dt.tree_.feature[0]].split("=")[0]

'model_year'

**Answer:** `model_year`

## Question 2

Random forest with `n_estimators=10`, `random_state=1`. RMSE on validation?

In [9]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1).fit(X_train, y_train)
round(rmse(y_val, rf.predict(X_val)), 4)

np.float64(1.8394)

**Answer:** 1.837 (closest option to 1.8394)

## Question 3

Try `n_estimators` in `[10, 50, 100, 150]`. Which gives the lowest validation RMSE (3 decimal places)?

In [10]:
scores = {}
for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1).fit(X_train, y_train)
    scores[n] = round(rmse(y_val, rf.predict(X_val)), 3)

pd.Series(scores, name="rmse")

10     1.839
50     1.773
100    1.768
150    1.768
Name: rmse, dtype: float64

**Answer:** 100 (100 and 150 tie at 1.768; 100 reaches it first)

## Question 4

For each `max_depth` in `[10, 15, 20, 25]`, average the validation RMSE over `n_estimators` in `[10, 50, 100, 150]`.

In [11]:
rows = []
for d in [10, 15, 20, 25]:
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(n_estimators=n, max_depth=d, random_state=1, n_jobs=-1).fit(X_train, y_train)
        rows.append((d, n, rmse(y_val, rf.predict(X_val))))

df_scores = pd.DataFrame(rows, columns=["max_depth", "n_estimators", "rmse"])
df_scores.groupby("max_depth").rmse.mean().round(4)

max_depth
10    1.7541
15    1.7837
20    1.7878
25    1.7867
Name: rmse, dtype: float64

**Answer:** 10

## Question 5

Random forest with `n_estimators=10`, `max_depth=20`, `random_state=1`. Most important feature among the four candidates?

In [12]:
rf = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1).fit(X_train, y_train)
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
importances.round(4)

model_year                      0.2959
vehicle_weight                  0.2031
fuel_type=Gasoline              0.1607
horsepower                      0.0785
engine_displacement             0.0612
acceleration                    0.0553
origin=USA                      0.0423
drivetrain=Front-wheel drive    0.0201
num_doors                       0.0194
fuel_type=Diesel                0.0180
fuel_type=Hybrid                0.0148
drivetrain=All-wheel drive      0.0080
num_cylinders                   0.0072
origin=Asia                     0.0064
drivetrain=Rear-wheel drive     0.0056
origin=Europe                   0.0036
dtype: float64

In [13]:
importances[["vehicle_weight", "horsepower", "acceleration", "engine_displacement"]]

vehicle_weight         0.203067
horsepower             0.078468
acceleration           0.055342
engine_displacement    0.061152
dtype: float64

**Answer:** `vehicle_weight` (`model_year` ranks higher overall but isn't an option)

## Question 6

XGBoost for 100 rounds with `eta=0.3`, then `eta=0.1`. Which gives the better validation RMSE?

In [14]:
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)
watchlist = [(dtrain, "train"), (dval, "val")]

In [15]:
scores = {}
for eta in [0.3, 0.1]:
    xgb_params = {
        "eta": eta,
        "max_depth": 6,
        "min_child_weight": 1,
        "objective": "reg:squarederror",
        "nthread": 8,
        "seed": 1,
        "verbosity": 1,
    }
    model = xgb.train(xgb_params, dtrain, num_boost_round=100, evals=watchlist, verbose_eval=False)
    scores[eta] = round(rmse(y_val, model.predict(dval)), 4)

scores

{0.3: np.float64(1.8265), 0.1: np.float64(1.7248)}

**Answer:** 0.1